# 02 Where collection fails: news record and resident survey

Questions: in which Lagos LGAs does the news record report household waste collection failing, what problems does it describe, and whom does it hold responsible? How do the experiences reported by surveyed residents compare?

Inputs: `data/processed/complaints.csv`, built by `python -m lagos_waste.tagging build` from 213 BusinessDay and News Agency of Nigeria articles collected for 2021 to 2026 and tagged under `docs/tagging_protocol.md`. Tags on a stratified sample of 50 articles were re-coded independently against the full text, and corrected values replace the model tags before the file is built. LGA boundaries are GRID3 LGA boundaries [S16].

A news article is counted as a service-failure report when it is tagged `service_failure_report` or describes missed collection (`is_service_failure = yes`). Counts reflect where outlets reported failure, not where failure is most frequent. The resident survey [S17] (`data/processed/survey_clean.csv`, built by `python -m lagos_waste.survey`) records failure as experienced by households; it is a pilot convenience sample of 28 responses, so its results are reported as counts and are not broken down by LGA.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

import pandas as pd

from lagos_waste import charts, geo
from lagos_waste import news_analysis as na
from lagos_waste.survey import CLEAN_PATH
from lagos_waste.paths import FIGURES, MAPS

pd.set_option("display.max_colwidth", 80)
SOURCE_NEWS = "Source: BusinessDay and NAN articles, 2021 to 2026, tagged by the author's protocol (complaints.csv)."
SOURCE_MAP = SOURCE_NEWS + " Boundaries: GRID3 via geoBoundaries [S16]."
SOURCE_SURVEY = "Source: resident survey, pilot convenience sample, n = 28, 3 to 4 October 2026 [S17]."


## Load

In [2]:
complaints = na.load_complaints()
arts = na.articles(complaints)
failure = arts[arts["is_service_failure"] == "yes"]
print(f"{len(arts)} relevant articles ({arts['outlet'].value_counts().to_dict()}), "
      f"{arts['date'].min():%d %b %Y} to {arts['date'].max():%d %b %Y}")
print(f"{len(failure)} service-failure articles; {len(complaints)} article-LGA rows")

84 relevant articles ({'businessday': 56, 'nannews': 28}), 14 Feb 2021 to 26 Sep 2026
21 service-failure articles; 173 article-LGA rows


## 1. What the news record covers

Articles by publication year and type. Coverage is concentrated in 2026, when the state's waste backlog became a public issue; earlier years hold few articles of any type.

In [3]:
by_year = na.type_by_year(complaints)
by_year["total"] = by_year.sum(axis=1)
by_year

article_type,enforcement,official_statement,opinion,other,policy_or_investment,service_failure_report,total
year,,,,,,,
2021,0,1,1,0,3,0,5
2022,3,5,1,0,2,2,13
2023,3,0,0,1,1,1,6
2024,0,3,0,1,2,1,7
2025,2,2,1,1,5,0,11
2026,4,22,3,4,2,7,42


In [4]:
print(failure["date"].dt.year.value_counts().sort_index().to_string())
print()
print(failure["article_type"].value_counts().to_string())

date
2022     3
2023     1
2024     2
2025     1
2026    14

article_type
service_failure_report    11
official_statement         4
opinion                    2
other                      2
policy_or_investment       1
enforcement                1


## 2. Service-failure reports by LGA

`failure_articles` counts distinct articles that place a service failure in the LGA. An article naming several LGAs counts once in each. `failure_lga_weeks` counts distinct calendar weeks with at least one such article, so several reports of one event in one week count once [A07]. `Lagos-wide` and `unspecified` rows hold articles without a placeable LGA.

In [5]:
by_lga = na.counts_by_lga(complaints)
by_lga

,lga,articles,failure_articles,failure_lga_weeks
0,Alimosho,18,8,8
1,Ajeromi-Ifelodun,10,8,8
2,Oshodi-Isolo,10,6,6
3,Ikorodu,7,5,5
4,Ikeja,11,4,4
5,Mushin,11,4,4
6,Agege,7,4,4
7,Kosofe,5,3,3
8,Surulere,4,3,3
9,Amuwo-Odofin,6,2,2


In [6]:
differ = by_lga[by_lga["failure_articles"] != by_lga["failure_lga_weeks"]]
print(f"LGAs where the LGA-week count differs from the article count: {len(differ)}")
lgas_only = by_lga[by_lga["lga"].isin(geo.LGAS)]
print(f"LGAs with at least one service-failure article: {(lgas_only['failure_articles'] > 0).sum()} of 20")

LGAs where the LGA-week count differs from the article count: 0
LGAs with at least one service-failure article: 17 of 20


In [7]:
shown = lgas_only[lgas_only["articles"] > 0]
charts.grouped_hbar(
    list(shown["lga"]),
    [("Service-failure articles", list(shown["failure_articles"])), ("All relevant articles", list(shown["articles"]))],
    "News articles placing a waste problem in each LGA, 2021 to 2026",
    "Articles",
    SOURCE_NEWS,
    FIGURES / "02_news_articles_by_lga.png",
);

## 3. Map of service-failure reports

Static map saved to `reports/figures/02_map_failure_articles.png`; interactive version saved to `reports/maps/02_failure_articles.html`.

In [8]:
lga_shapes = geo.load_lagos_lgas()
mapped = geo.join_counts(lga_shapes, by_lga, "failure_articles")
geo.choropleth_png(
    mapped, "failure_articles",
    "News articles reporting collection failure, by LGA, 2021 to 2026",
    "Service-failure articles", SOURCE_MAP, FIGURES / "02_map_failure_articles.png",
)
geo.choropleth_html(mapped, "failure_articles", "Service-failure articles, 2021 to 2026", SOURCE_MAP,
                    MAPS / "02_failure_articles.html");

## 4. Problems described

Share of articles describing each problem type, for service-failure articles and for all relevant articles. `missed_collection` is close to 100% among service-failure articles by construction, because it is one of the two conditions that define them.

In [9]:
fail_p = na.problem_shares(complaints, failure_only=True).set_index("problem_type")
all_p = na.problem_shares(complaints).set_index("problem_type")
problems = fail_p.join(all_p, lsuffix="_failure", rsuffix="_all")
problems

,articles_failure,share_pct_failure,articles_all,share_pct_all
problem_type,,,,
missed_collection,20,95.2,20,23.8
illegal_dumping,16,76.2,52,61.9
waste_pileup,15,71.4,28,33.3
fees_or_billing,10,47.6,13,15.5
operator_capacity,10,47.6,23,27.4
dumpsite_access,7,33.3,11,13.1
blocked_drains,6,28.6,12,14.3
road_access,4,19.0,4,4.8
market_waste,2,9.5,9,10.7


In [10]:
charts.grouped_hbar(
    [p.replace("_", " ").capitalize() for p in problems.index],
    [("Service-failure articles", list(problems["share_pct_failure"])), ("All relevant articles", list(problems["share_pct_all"]))],
    "Problems described in news articles on waste collection, 2021 to 2026",
    "Share of articles describing the problem (%)",
    SOURCE_NEWS,
    FIGURES / "02_news_problem_types.png",
    value_format="{:.0f}%",
    xmax=105,
);

## 5. Whom the news holds responsible

Articles assigning blame to each actor, by who assigns it. `mixed` means the article reports blame from more than one kind of speaker. An article can blame several actors.

In [11]:
blame = na.blame_by_source(complaints)
blame

blame_source,expert,journalist,mixed,official,resident,total
blamed,,,,,,
residents,1,2,9,24,2,38
psp_operators,1,1,9,7,3,21
traders,0,1,2,8,0,11
cart_pushers,0,0,2,7,0,9
state_government,0,2,3,0,0,5
lawma,0,0,4,0,0,4
other,0,0,1,2,0,3
local_government,0,0,1,0,0,1


In [12]:
official = blame.get("official", 0)
other = blame["total"] - official
charts.grouped_hbar(
    [{"psp_operators": "PSP operators", "lawma": "LAWMA"}.get(b, b.replace("_", " ").capitalize()) for b in blame.index],
    [("Blame stated by officials", list(official)), ("Blame stated by residents, journalists, experts or mixed", list(other))],
    "Actors held responsible in news articles on waste collection, 2021 to 2026",
    "Articles",
    SOURCE_NEWS,
    FIGURES / "02_news_blame.png",
);

In [13]:
fail_blame = failure.assign(blamed=failure["blamed"].str.split(";")).explode("blamed")["blamed"].value_counts()
fail_blame.rename("service_failure_articles")

blamed
psp_operators       14
residents           13
state_government     4
lawma                4
none_stated          3
cart_pushers         2
traders              2
other                1
local_government     1
Name: service_failure_articles, dtype: int64

## 6. Resident survey: who responded

Twenty-eight responses were received on 3 and 4 October 2026; none was excluded for missing consent or duplication (`data/interim/survey_exclusions.csv`). Respondents come from 14 LGAs, with 8 in Alimosho and no more than 4 in any other LGA, so LGA comparisons are not made.

In [14]:
survey = pd.read_csv(CLEAN_PATH, dtype=str, keep_default_na=False)
print(f"{len(survey)} responses, submitted {survey['submitted_date'].min()} to {survey['submitted_date'].max()}")
for col in ["lga", "road_condition", "home_type", "gated_estate", "main_collector", "pays_bill"]:
    print(f"\n{col}:", survey[col].value_counts().to_dict())

28 responses, submitted 2026-10-03 to 2026-10-04

lga: {'Alimosho': 8, 'Ifako-Ijaiye': 4, 'Mushin': 2, 'Oshodi-Isolo': 2, 'Lagos Island': 2, 'Agege': 2, 'Ajeromi-Ifelodun': 1, 'Surulere': 1, 'Apapa': 1, 'Ikorodu': 1, 'Badagry': 1, 'Ibeju-Lekki': 1, 'Epe': 1, 'Eti-Osa': 1}

road_condition: {'tarred_good': 15, 'untarred': 8, 'tarred_damaged': 5}

home_type: {'flat_2plus': 12, 'self_contained': 9, 'shared_room': 6, 'other': 1}

gated_estate: {'yes': 18, 'no': 10}

main_collector: {'psp_truck': 19, 'cart_pusher': 4, 'burn_bury': 2, 'dump': 1, 'estate_landlord': 1, 'dont_know': 1}

pays_bill: {'regularly': 21, 'no': 4, 'sometimes': 2, 'dont_know': 1}


## 7. Resident survey: collection service

`gap_over_14_days` is `yes` when the last collection by anyone was more than 14 days ago or never happened, and blank when the respondent could not remember. `missed_pickups_4wk` compares stated truck schedule with truck collections in the last 4 weeks [A08]; it is blank for households without a fixed schedule or that did not know the count.

In [15]:
truck = survey[survey["truck_user"] == "yes"]
known_missed = truck[truck["missed_pickups_4wk"] != ""]
print(f"Households mainly served by the PSP or LAWMA truck: {len(truck)} of {len(survey)}")
print("Stated truck schedule:", truck["truck_schedule"].value_counts().to_dict())
print(f"Truck households with at least one missed collection in 4 weeks: "
      f"{(known_missed['missed_pickups_4wk'].astype(int) > 0).sum()} of {len(known_missed)} with a known schedule and count")
print("Last collection more than 14 days ago (all respondents):", survey["gap_over_14_days"].replace("", "cannot remember").value_counts().to_dict())
coping = {c.replace("if_missed_", ""): int((survey[c] == "yes").sum()) for c in survey.columns if c.startswith("if_missed_")}
print("When the truck does not come:", coping)
print(f"Mean satisfaction (1 to 5): {survey['satisfaction'].astype(int).mean():.1f}")

Households mainly served by the PSP or LAWMA truck: 19 of 28
Stated truck schedule: {'fortnightly': 6, 'no_schedule': 6, 'monthly': 4, 'weekly': 3}
Truck households with at least one missed collection in 4 weeks: 5 of 11 with a known schedule and count
Last collection more than 14 days ago (all respondents): {'no': 11, 'yes': 11, 'cannot remember': 6}
When the truck does not come: {'wait': 9, 'call': 5, 'cart_pusher': 9, 'burn': 3, 'dump': 3, 'collection_point': 2, 'other': 1}
Mean satisfaction (1 to 5): 2.4


## 8. Resident survey: first look at the explanations

Each table cross-tabulates one explanation with `gap_over_14_days`. With 28 responses, these tables show direction only; no significance tests are reported.

In [16]:
def gap_table(frame: pd.DataFrame, by: str) -> pd.DataFrame:
    """Counts of last-collection gap by a grouping column, with row totals."""
    gap = frame["gap_over_14_days"].replace({"yes": "over 14 days", "no": "14 days or less", "": "cannot remember"})
    return pd.crosstab(frame[by], gap, margins=True, margins_name="total")

print("H1 payment (all respondents)")
display(gap_table(survey, "pays_bill"))
paying_truck = truck[truck["pays_bill"] == "regularly"]
known = paying_truck[paying_truck["gap_over_14_days"] != ""]
print(f"Regular payers served by the truck whose last collection was over 14 days ago: "
      f"{(known['gap_over_14_days'] == 'yes').sum()} of {len(known)} who remembered")

H1 payment (all respondents)


gap_over_14_days,14 days or less,cannot remember,over 14 days,total
pays_bill,,,,
dont_know,0,1,0,1
no,1,0,3,4
regularly,9,4,8,21
sometimes,1,1,0,2
total,11,6,11,28


Regular payers served by the truck whose last collection was over 14 days ago: 6 of 13 who remembered


In [17]:
print("H3 road condition")
display(gap_table(survey, "road_condition"))
print("H5 home type and gated estate")
display(gap_table(survey, "home_type"))
display(gap_table(survey, "gated_estate"))
nonpay = survey[survey["pays_bill"].isin(["no", "sometimes"])]
print("Reasons given by the", len(nonpay), "households that do not pay or pay sometimes:",
      {c.replace("nonpay_", ""): int((nonpay[c] == "yes").sum()) for c in survey.columns if c.startswith("nonpay_")})

H3 road condition


gap_over_14_days,14 days or less,cannot remember,over 14 days,total
road_condition,,,,
tarred_damaged,2,1,2,5
tarred_good,7,5,3,15
untarred,2,0,6,8
total,11,6,11,28


H5 home type and gated estate


gap_over_14_days,14 days or less,cannot remember,over 14 days,total
home_type,,,,
flat_2plus,6,1,5,12
other,1,0,0,1
self_contained,3,2,4,9
shared_room,1,3,2,6
total,11,6,11,28


gap_over_14_days,14 days or less,cannot remember,over 14 days,total
gated_estate,,,,
no,5,3,2,10
yes,6,3,9,18
total,11,6,11,28


Reasons given by the 6 households that do not pay or pay sometimes: {'not_applicable': 0, 'poor_service': 2, 'cannot_afford': 0, 'no_bill': 1, 'dont_know_who': 1, 'pays_cart_pusher': 1, 'other': 1}


In [18]:
road = gap_table(survey, "road_condition").drop(index="total").reindex(["tarred_good", "tarred_damaged", "untarred"])
charts.grouped_hbar(
    ["Tarred, good condition", "Tarred but badly damaged", "Untarred"],
    [("Last collection over 14 days ago", list(road["over 14 days"])),
     ("Last collection 14 days ago or less", list(road["14 days or less"]))],
    "Time since last collection by road condition, surveyed households",
    "Households",
    SOURCE_SURVEY + " Respondents who could not remember are not shown.",
    FIGURES / "02_survey_gap_by_road.png",
);

## Findings

1. The tagged news record holds 84 relevant articles after duplicates are merged (56 BusinessDay, 28 NAN), of which 21 report collection failure. Fourteen of the 21 were published in 2026.
2. Service failure is placed in 17 of the 20 LGAs. Alimosho and Ajeromi-Ifelodun have the most service-failure articles (8 each), followed by Oshodi-Isolo (6) and Ikorodu (5). Epe, Ojo and Ifako-Ijaiye have none. Counting LGA-weeks instead of articles changes no LGA's count, so repeated coverage of single events does not drive the ranking [A07].
3. Across all relevant articles, illegal dumping is the most frequently described problem (62% of articles). Among service-failure articles, missed collection (by construction), illegal dumping (76%) and refuse heaps (71%) lead; fees and billing and operator capacity each appear in 48%, dumpsite access in 33% and road access in 19%.
4. Officials are the source of blame in 35 of the 56 articles that assign it, and they mostly blame residents (24 articles), traders (8) and cart pushers (7). In service-failure articles, where blame comes mostly from residents or from several kinds of speaker, PSP operators are blamed as often as residents (14 and 13 articles). LAWMA is blamed only in service-failure articles (4), and the state government in 4 of the 5 articles that blame it.

5. In the resident survey, 21 of 28 respondents say they pay their waste bill regularly. Among regular payers mainly served by the PSP or LAWMA truck, 6 of 13 who could remember had last been served more than 14 days earlier. Paying regularly did not secure regular collection for these households, which runs against non-payment as the main explanation (H1).
6. Only 3 of the 19 truck households report a weekly schedule, and 6 report no fixed schedule. Of the 11 with a known schedule and count, 5 had at least one missed collection in the last 4 weeks. When the truck does not come, 9 households wait and 9 pay a cart pusher.
7. Six of the 8 respondents on untarred roads had last been served more than 14 days earlier, compared with 3 of the 10 on good tarred roads who remembered, which is consistent with road access (H3). Respondents in gated estates reported long gaps more often than others (9 of 15 against 2 of 7), so the home-type proxy for income (H5) shows no clear pattern in this sample.
8. Mean satisfaction is 2.4 on a 1 to 5 scale: 1.5 among households with a gap of more than 14 days and 3.5 among the rest. None of the 6 households that do not pay or pay only sometimes gave cost as a reason; 2 cited poor service.

## Limitations

- Two outlets were collected; Guardian, Punch, Vanguard and Peoples Gazette blocked automated access (`data/interim/news_scrape_log.csv`). LGA counts therefore reflect the reporting patterns of BusinessDay and NAN.
- Reports cluster in LGAs that journalists visit and that residents' groups bring to the press; low counts do not show good service.
- An article naming several LGAs adds one to each, including LGAs named only for a dumpsite or a quotation.
- Tags were produced by a language model under a written protocol and checked on a sample of 50 articles (agreement 90% to 98% per field).
- The survey is a pilot convenience sample of 28 responses collected through the author's networks over two days. Most respondents live on tarred roads (20 of 28) and in gated estates (18 of 28); the sample cannot be generalised to Lagos, and supports direction-of-effect statements only.